In [1]:
import openai
import instructor
from qdrant_client import QdrantClient

from pydantic import BaseModel, Field

In [2]:
client = instructor.from_openai(openai.OpenAI())

In [4]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")

In [5]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-item-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context



def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role": "system", "content": prompt}],
        temperature=0,
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "datamodel": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [7]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [8]:
output = rag_pipeline("Can I get a charging cable? Please suggest me a good one.", qdrant_client)

In [9]:
output

{'datamodel': RAGGenerationResponse(answer='Based on the available products, I suggest the GREPHONE 2 Pack USB C to Lightning Cable as a good charging cable option. Here are the detailed specifications:\n\n- Length: 6 feet (extra long for convenience)\n- Certification: Apple MFi Certified for guaranteed compatibility and safety\n- Charging Speed: Supports PD Fast Charge 3A/30W (max) for fast charging\n- Compatibility: Works with iPhone 13/12/11 series, iPad models including iPad Pro 12.9" gen1/gen2, iPad Air3, iPad mini5, and more\n- Design: Reinforced material for durability, 3D aluminum joint, laser welding technology to prevent breakage\n- Features: Supports fast and stable charging, automatic chip recognition, overvoltage protection\n- Package: Comes in a 2 pack\n- Warranty: 12-month service and customer support\n\nThis cable is ideal for fast charging and durability, making it a reliable choice for your charging needs.\n\nUsed chunk ID: B0BV6PWVCG'),
 'answer': 'Based on the avail

In [11]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="The ID of the item used to answer the question")
    description: str = Field(description="Short description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [12]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-item-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context



def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role": "system", "content": prompt}],
        temperature=0,
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "original_output": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [13]:
result = rag_pipeline("Can I get some earphones?", qdrant_client, top_k=10)

In [14]:
result

{'original_output': RAGGenerationResponse(answer='Yes, you can get several types of earphones from the available products. Here are some options with detailed specifications:\n\n1. Wireless Earbuds S23-vine earbuds (ID: B0B9FTVL58)\n- Bluetooth 5.3\n- 37 hours playback time\n- LED power display\n- In-ear design with deep bass\n- IPX7 waterproof\n- Ultra-light with charging case\n- Smart touch controls\n- Suitable for sports\n\n2. TUNEAKE Kids Headphones (ID: B0C142QS8X)\n- Over-ear design\n- Volume limited to 94dB for hearing protection\n- Foldable and adjustable headband\n- 3.5mm jack compatibility\n- No microphone\n- Comfortable padded headband and soft ear cups\n- Compatible with smartphones, tablets, laptops\n\n3. TELSOR Wireless Earbuds (ID: B0C6K1GQCF)\n- Bluetooth 5.1\n- 30 hours playtime with charging case\n- IPX7 waterproof\n- Noise cancelling mic for calls\n- Touch control for volume, play, calls\n- Compatible with iPhone, Android, smart TVs, computers\n\n4. Open Ear Headphon

In [15]:
print(result["answer"])

Yes, you can get several types of earphones from the available products. Here are some options with detailed specifications:

1. Wireless Earbuds S23-vine earbuds (ID: B0B9FTVL58)
- Bluetooth 5.3
- 37 hours playback time
- LED power display
- In-ear design with deep bass
- IPX7 waterproof
- Ultra-light with charging case
- Smart touch controls
- Suitable for sports

2. TUNEAKE Kids Headphones (ID: B0C142QS8X)
- Over-ear design
- Volume limited to 94dB for hearing protection
- Foldable and adjustable headband
- 3.5mm jack compatibility
- No microphone
- Comfortable padded headband and soft ear cups
- Compatible with smartphones, tablets, laptops

3. TELSOR Wireless Earbuds (ID: B0C6K1GQCF)
- Bluetooth 5.1
- 30 hours playtime with charging case
- IPX7 waterproof
- Noise cancelling mic for calls
- Touch control for volume, play, calls
- Compatible with iPhone, Android, smart TVs, computers

4. Open Ear Headphones (ID: B0CBMPG524)
- Bluetooth 5.3
- 60 hours playtime with charging case
- IP